# 06 — Gộp observation thu qua router (MQTT) vào dataset train

Router đẩy observation qua MQTT, `server/mqtt_listener.py` ghi vào
`server/data/device_observations.jsonl`: **mỗi dòng = một cửa sổ của một MAC**, gồm
DHCP / DNS / mDNS / TLS ClientHello đã bóc sẵn (không có pcap). Notebook này biến các
dòng đó thành session đúng lược đồ `sessions.parquet`, cùng hàm `aggregate()` với 04.

Các bước:

1. Đọc mọi nguồn JSONL, loại trùng theo `message_id`, chốt **snapshot** có sha256 —
   listener vẫn đang ghi thêm vào file gốc.
2. Observation → record chuẩn → `aggregate()`. **Golden test**: 9 cột chung phải khớp
   hệt adapter lúc chạy thật (`Code/test_release_observations.py::feature_row`).
3. **Tự gán nhãn** theo MAC, không cần xác nhận tay: nhãn FIELD cũ trùng hostname → luật
   hostname `device_list.csv` → máy Windows (DHCP `MSFT`) nhận nhãn chung. MAC không đủ
   thông tin bị bỏ, có ghi lý do. Nhãn đã dùng ghi ra `Data/router_obs/device_labels_router.csv`;
   muốn sửa một dòng thì điền tay và đặt `verified=1` (tuỳ chọn).
4. Làm sạch: bỏ cửa sổ < `MIN_SOURCES` nguồn (runtime luôn abstain với chúng).
5. Chia train/test **theo thiết bị** (không theo MAC, không theo cửa sổ), split được
   đóng băng trong `device_split.csv`. Thiết bị đã có trong FIELD pcap luôn ở train.
   Train giới hạn `MAX_WINDOWS_PER_DEVICE` cửa sổ/thiết bị để một máy không lấn át.
6. Ghi bộ test giữ riêng (raw JSONL + parquet 12 cột + nhãn) và **tuỳ chọn** đo một
   model release trên đó.
7. `APPLY = True` mới ghi vào `Data/sessions.parquet`, `device_labels_verified.csv`
   và dựng lại `sessions_verified.parquet` (có backup trước khi ghi). Kèm theo
   `dhcp_features_router.csv` (hostname expert) và `device_os_router.csv` (head OS) — 05
   tự đọc hai file này nếu có.

Session router mang `scenario = "FIELD"` (05 coi FIELD là lớp thật) và
`capture_file = "router/<message_id>"`; 04 không xoá các dòng có tiền tố này khi chạy lại.
Chạy lại notebook này thì các dòng `router/` cũ được thay bằng bản mới — không nhân đôi.

In [ ]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [ ]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

In [ ]:
# --- Đường dẫn dự án ---
from pathlib import Path


DATA = ROOT / "Data"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

# pcap tự thu (điện thoại, laptop... trong mạng nhà). Tên thư mục khác nhau giữa server
# (`data_pcap`) và máy local (`data_pcap_capture`) — một chỗ quyết định cho mọi bước.
CAPTURE_DIR = next((DATA / name for name in ("data_pcap", "data_pcap_capture")
                    if (DATA / name).is_dir()), DATA / "data_pcap")

In [ ]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)

RARE_THRESHOLD = 10      # lớp dưới ngưỡng này không học được, tách riêng khi report


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 03_build_dataset và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 03_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None


MAX_TOKENS = 40
TOKEN_SPLIT = re.compile(r"[.\-]")

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def normalize_cipher_suites(value):
    if _is_missing(value):
        return MISSING
    values = value if isinstance(value, (list, tuple)) else re.findall(r"\d+", str(value))
    normalized = []
    for item in values:
        try:
            number = int(item)
        except (TypeError, ValueError):
            continue
        if not is_grease(number):
            normalized.append(str(number))
    return ",".join(normalized) if normalized else MISSING


def normalize_tls_version(value):
    if _is_missing(value):
        return MISSING
    text = str(value).strip().lower()
    if text.startswith("0x"):
        text = text[2:]
    if re.fullmatch(r"\d{1,4}", text):
        return text.zfill(4)
    return MISSING

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def domain_tokens(names, max_tokens=MAX_TOKENS):
    """Domain names to unique lowercase tokens, preserving first-seen order."""
    tokens = []
    for name in names:
        if _is_missing(name):
            continue
        normalized = str(name).lower().rstrip(".")
        tokens.extend(token for token in TOKEN_SPLIT.split(normalized) if token)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


def _mode(values):
    clean = [str(value) for value in values
             if not _is_missing(value) and str(value).strip() not in ("", MISSING)]
    if not clean:
        return MISSING
    return pd.Series(clean).mode().iat[0]


def _tls_fingerprint(record):
    version = normalize_tls_version(record.get("version", record.get("tls_version")))
    ciphers = normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
    alpn = _text(record.get("alpn"), lowercase=True) or ""
    if MISSING in (version, ciphers):
        return MISSING
    return f"{version}|{ciphers}|{alpn}"

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def aggregate(records, feature_cols):
    """Aggregate canonical records for one device/window into one model feature row."""
    by_source = {source: [] for source in RAW_FIELDS}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("Each raw record must be a dictionary")
        proto = record.get("proto")
        if proto not in by_source:
            raise ValueError(f"Invalid proto {proto!r}; expected one of {sorted(RAW_FIELDS)}")
        by_source[proto].append(record)

    row = {}
    row["dhcp_prl"] = _mode(record.get("opt55", record.get("param_req_list"))
                             for record in by_source["dhcp"])
    row["dhcp_vci"] = _mode(record.get("opt60", record.get("vendor_class_id"))
                             for record in by_source["dhcp"])
    options = set()
    if row["dhcp_prl"] != MISSING:
        options = {int(value) for value in re.findall(r"\d+", row["dhcp_prl"])}
    for column in feature_cols:
        if column.startswith("dhcp_opt_"):
            row[column] = int(int(column.rsplit("_", 1)[1]) in options)
    row["dhcp_prl_len"] = len(options)

    row["dns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                         for record in by_source["dns"])
    row["mdns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                          for record in by_source["mdns"])

    tls_records = by_source["tls"]
    row["tls_fp"] = _mode(_tls_fingerprint(record) for record in tls_records)
    row["tls_version"] = _mode(
        normalize_tls_version(record.get("version", record.get("tls_version")))
        for record in tls_records
    )
    row["tls_alpn"] = _mode(_text(record.get("alpn"), lowercase=True)
                              for record in tls_records)
    row["tls_ciphers"] = _mode(
        normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
        for record in tls_records
    )
    row["tls_sni_tokens"] = domain_tokens(record.get("sni") for record in tls_records)

    for source in RAW_FIELDS:
        row[f"has_{source}"] = int(bool(by_source[source]))
    row["n_sources"] = sum(row[flag] for flag in SOURCE_FLAGS)

    numeric_prefixes = ("has_", "dhcp_opt_")
    for column in feature_cols:
        if column not in row:
            row[column] = (0 if column.startswith(numeric_prefixes)
                           or column == "dhcp_prl_len" else MISSING)
    return row

In [ ]:
# --- Dựng dataset đã xác minh ---
import argparse
from pathlib import Path

import pandas as pd


# Tên có tiền tố `DEFAULT_` vì sau `%run` mọi module nằm chung một namespace: `SOURCE`
# hay `OUTPUT` trần là tên quá dễ đụng với module khác.
DEFAULT_SESSIONS = DATA / "sessions.parquet"
DEFAULT_LABELS = DATA / "device_labels_verified.csv"
DEFAULT_EXCLUDED = DATA / "device_labels_excluded.csv"
DEFAULT_OUTPUT = SESSIONS_PATH


def _read_csv(path: Path, required: set[str]) -> pd.DataFrame:
    frame = pd.read_csv(path, keep_default_na=False)
    frame = frame.rename(columns={old: new for old, new in HEAD_ALIASES.items()
                                  if old in frame.columns and new not in frame.columns})
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    return frame


def validate_taxonomy(labels: pd.DataFrame, excluded: pd.DataFrame) -> None:
    required = ["canonical_device", *LABEL_COLS]
    if labels[required].apply(lambda col: col.astype(str).str.strip().eq("")).any().any():
        raise ValueError("verified labels contain blank required values")
    if labels["canonical_device"].duplicated().any():
        duplicates = sorted(labels.loc[labels["canonical_device"].duplicated(False),
                                           "canonical_device"].unique())
        raise ValueError(f"duplicate verified devices: {duplicates}")
    if excluded["canonical_device"].duplicated().any():
        raise ValueError("excluded device list contains duplicates")
    overlap = set(labels["canonical_device"]) & set(excluded["canonical_device"])
    if overlap:
        raise ValueError(f"devices cannot be both verified and excluded: {sorted(overlap)}")
    unknown = labels["make"].astype(str).str.casefold().eq("unknown")
    if unknown.any():
        raise ValueError("make=Unknown is forbidden in the production taxonomy")

    model_make = labels.groupby("model")["make"].nunique()
    conflicts = model_make[model_make.ne(1)]
    if not conflicts.empty:
        raise ValueError(f"model maps to multiple makes: {conflicts.index.tolist()}")
    model_type = labels.groupby("model")["type"].nunique()
    conflicts = model_type[model_type.ne(1)]
    if not conflicts.empty:
        raise ValueError(f"model maps to multiple types: {conflicts.index.tolist()}")


def build_verified_dataset(
    source: Path = DEFAULT_SESSIONS,
    labels_path: Path = DEFAULT_LABELS,
    excluded_path: Path = DEFAULT_EXCLUDED,
    output: Path = DEFAULT_OUTPUT,
) -> pd.DataFrame:
    labels = _read_csv(labels_path, {"canonical_device", *LABEL_COLS})
    excluded = _read_csv(excluded_path, {"canonical_device", "reason"})
    validate_taxonomy(labels, excluded)

    sessions = pd.read_parquet(source)
    known = set(labels["canonical_device"])
    dropped = set(excluded["canonical_device"])
    present = set(sessions["canonical_device"].astype(str))
    unclassified = present - known - dropped
    if unclassified:
        raise ValueError(f"sessions contain unclassified devices: {sorted(unclassified)}")
    unused = known - present
    if unused:
        raise ValueError(f"verified labels contain devices absent from sessions: {sorted(unused)}")

    result = sessions[~sessions["canonical_device"].isin(dropped)].copy()
    # Nhãn luôn được gán lại từ taxonomy ở ngay dưới, nên cột nhãn mang tên cũ trong file
    # nguồn chỉ là tàn dư. Giữ lại thì dataset có cả `family` lẫn `model` cùng lúc.
    result = result.drop(columns=[c for c in HEAD_ALIASES if c in result.columns])
    mapping = labels.set_index("canonical_device")[LABEL_COLS]
    for column in LABEL_COLS:
        result[column] = result["canonical_device"].map(mapping[column])
    if result[LABEL_COLS].isna().any().any():
        raise ValueError("label mapping unexpectedly produced missing values")

    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_parquet(output)
    print(
        f"wrote {output}: {len(result)} sessions, "
        f"{result.canonical_device.nunique()} devices; excluded={sorted(dropped)}"
    )
    return result

In [ ]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Hằng của contract `sdc_iden.onnx` (09). 12/13 ghép head OS vào chính graph này nên dùng chung.
import hashlib
UNKNOWN_LABEL = "__unknown__"
NOT_PRESENT = ("", MISSING, "nan", "None")

In [ ]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Chuẩn bị input 12 cột — bản Python của phần graph tự tính (`build_derived` ở 09).
import re

import numpy as np
import pandas as pd


_MAC_RE = re.compile(r"^[0-9a-f]{2}(:[0-9a-f]{2}){5}$")


def canonical_mac(value):
    if value is None:
        return MISSING
    text = str(value).strip().lower().replace("-", ":")
    return text if _MAC_RE.match(text) else MISSING


def mac_is_random(value):
    """Bit U/L (0x02) của octet đầu: 1 = locally administered = MAC ngẫu nhiên hoá."""
    mac = canonical_mac(value)
    return 0 if mac == MISSING else int(int(mac[:2], 16) & 0x02 != 0)

In [ ]:
import json
import shutil
import sys
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display


def rel(path):
    """Đường dẫn gọn để in; nằm ngoài dự án thì in nguyên."""
    try:
        return path.relative_to(ROOT)
    except ValueError:
        return path

## 1. Cấu hình

In [ ]:
OBS_SOURCES = [
    ROOT / "server" / "data" / "device_observations.jsonl",   # listener MQTT đang ghi
    DATA / "device_observations.jsonl",                        # lô cũ (16/10)
]
OBS_DIR = DATA / "router_obs"
LABELS_ROUTER = OBS_DIR / "device_labels_router.csv"   # bạn điền nhãn ở đây
SPLIT_PATH = OBS_DIR / "device_split.csv"              # split đóng băng theo thiết bị
ROUTER_DHCP_CSV = DATA / "features" / "dhcp_features_router.csv"
ROUTER_OS_CSV = OBS_DIR / "device_os_router.csv"

RAW_SESSIONS_PATH = DATA / "sessions.parquet"
LABELS_PATH = DATA / "device_labels_verified.csv"
EXCLUDED_PATH = DATA / "device_labels_excluded.csv"
VERIFIED_OUT = SESSIONS_PATH

CAPTURE_SCENARIO = "FIELD"
ROUTER_PREFIX = "router/"
MIN_SOURCES = 2                 # cửa sổ 1 nguồn: ngưỡng 1.01, runtime không bao giờ trả lời
MAX_WINDOWS_PER_DEVICE = 8      # chỉ áp cho train; test giữ đủ để bỏ phiếu mức thiết bị
TEST_FRAC = 0.3
SEED = 42
OS_CLASSES = ["iOS", "Android", "Windows", "macOS", "Linux", "Embedded"]   # khớp 05 phần B
VERIFIED_VALUES = {"1", "x", "y", "yes", "true"}   # dòng nhãn t ay (tuỳ chọn) ghi đè nhãn tự động
# Máy Windows (DHCP VCI "MSFT 5.0") không khớp luật hostname nào: nhãn chung, không đoán hãng.
# Đặt None để bỏ các máy này.
WINDOWS_FALLBACK = {"make": "Generic Laptop", "type": "Laptop", MODEL_HEAD: "Windows PC"}
# Nguồn nhãn đủ tin để đưa hostname sang hostname expert (05 phần C). Nhãn sinh từ chính luật
# hostname mà dạy lại expert hostname thì vòng tròn: expert chỉ học lại luật đó.
TRUSTED_SOURCES = {"nhãn tay", "FIELD verified"}

APPLY = True                # True = ghi vào dataset train (có backup)
EVAL_MODEL = None               # vd. ROOT / "Models/20261014_142321/release/sdc_os.onnx"

OBS_DIR.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

## 2. Đọc, loại trùng, chốt snapshot

Snapshot đặt tên theo sha256 nội dung: chạy lại khi dữ liệu chưa đổi thì không sinh file mới.

In [ ]:
observations, seen, per_source = [], set(), {}
for path in OBS_SOURCES:
    if not path.exists():
        print(f"bỏ qua (không có): {path}")
        continue
    added = dup = 0
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        obs = json.loads(line)
        mid = obs.get("message_id")
        assert mid, f"observation không có message_id trong {path}"
        if mid in seen:
            dup += 1
            continue
        seen.add(mid)
        obs["mac"] = canonical_mac(obs.get("mac"))
        observations.append(obs)
        added += 1
    per_source[str(rel(path))] = {"sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                                               "added": added, "duplicates": dup}

assert observations, "Không đọc được observation nào"
observations.sort(key=lambda o: o["message_id"])
snapshot_text = "".join(json.dumps(o, ensure_ascii=False, sort_keys=True) + "\n" for o in observations)
snapshot_sha = hashlib.sha256(snapshot_text.encode("utf-8")).hexdigest()
SNAPSHOT_PATH = OBS_DIR / f"observations_{snapshot_sha[:12]}.jsonl"
if not SNAPSHOT_PATH.exists():
    SNAPSHOT_PATH.write_text(snapshot_text, encoding="utf-8")
(OBS_DIR / "snapshot_manifest.json").write_text(json.dumps({
    "created": STAMP, "snapshot": str(rel(SNAPSHOT_PATH)), "sha256": snapshot_sha,
    "n_observations": len(observations), "sources": per_source,
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"{len(observations)} observation duy nhất — {len({o['mac'] for o in observations})} MAC")
display(pd.DataFrame(per_source).T)
print("snapshot:", rel(SNAPSHOT_PATH))

## 3. Observation → record chuẩn → `aggregate()` + golden test với adapter chạy thật

- DHCP: chỉ bản tin **của client** (có `parameter_request_list`). OFFER/ACK là của router —
  đưa vào thì `has_dhcp=1` trong khi lúc chạy thật graph thấy `dhcp_prl` trống (lệch train/serve).
- DNS/mDNS: tên trong `questions`, giống adapter.
- TLS: một record mỗi ClientHello; SNI thứ 2 trở đi thành record chỉ có `sni` (`_mode`
  bỏ qua giá trị thiếu nên không ảnh hưởng `tls_fp`).

In [ ]:
sys.path.insert(0, str(ROOT / "Code"))
import test_release_observations as serve   # adapter JSONL → 12 cột của lúc chạy thật

frame_ref, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]
SHARED = ["dhcp_prl", "dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens",
          "tls_fp", "tls_version", "tls_alpn", "tls_ciphers"]
SERVE_ONLY = [c for c in serve.COLUMNS if c not in SHARED]          # mac_oui, dhcp_hostname, mdns_model


def client_dhcp(obs):
    return [d for d in obs.get("dhcp", []) if d.get("parameter_request_list")]


def observation_records(obs):
    records = []
    for d in client_dhcp(obs):
        records.append({"proto": "dhcp",
                        "opt55": ",".join(str(v) for v in d["parameter_request_list"]),
                        "opt60": d.get("vendor_class_id") or None})
    for proto in ("dns", "mdns"):
        for entry in obs.get(proto, []):
            for q in entry.get("questions", []):
                records.append({"proto": proto, "qname": q.get("name")})
    for t in obs.get("tls", []):
        names = list(t.get("server_names") or [])
        records.append({"proto": "tls", "version": t.get("legacy_version"),
                        "ciphers": t.get("cipher_suites") or None,
                        "alpn": ",".join(t.get("alpn") or []) or None,
                        "sni": names[0] if names else None})
        records.extend({"proto": "tls", "sni": name} for name in names[1:])
    return records


def window_bounds(obs):
    stamps = sorted(x["timestamp"] for k in SOURCES for x in obs.get(k, []) if x.get("timestamp"))
    if not stamps:
        return None, None
    return pd.Timestamp(stamps[0]), pd.Timestamp(stamps[-1])


rows, mismatches = [], []
for obs in observations:
    agg = aggregate(observation_records(obs), features)
    srv = serve.feature_row(obs)
    for col in SHARED:
        if agg[col] != srv[col]:
            mismatches.append({"message_id": obs["message_id"], "column": col,
                               "aggregate": agg[col], "serve": srv[col]})
    start, end = window_bounds(obs)
    rows.append({
        "message_id": obs["message_id"], "mac": obs["mac"],
        "hostname": (obs.get("hostname") or "").strip(),
        "start": start, "end": end,
        "duration_s": (end - start).total_seconds() if start is not None else np.nan,
        **agg,
        **{f"serve_{c}": srv[c] for c in SERVE_ONLY},
    })

windows = pd.DataFrame(rows)
print(f"{len(windows)} cửa sổ, {len(features)} feature")
if mismatches:
    display(pd.DataFrame(mismatches).head(20))
assert not mismatches, (f"{len(mismatches)} ô lệch giữa aggregate() và adapter chạy thật — "
                        "sửa cho khớp trước khi train, nếu không model học một kiểu, chạy một kiểu")
print("golden test: 9 cột chung khớp 100% với test_release_observations.feature_row")

print("\nPhân bố n_sources:")
display(windows["n_sources"].value_counts().sort_index().rename("cửa sổ"))
print("Độ dài cửa sổ (s):")
display(windows["duration_s"].describe().round(1))

## 4. Tự gán nhãn theo MAC

Không cần xác nhận tay. Mỗi MAC lấy nhãn theo thứ tự ưu tiên, nguồn nào có trước thì dùng:

| # | Nguồn (`label_source`) | Cách lấy |
|---|---|---|
| 1 | `nhãn tay` | dòng bạn tự điền và đặt `verified=1` trong `device_labels_router.csv` (tuỳ chọn, để sửa nhãn tự động sai) |
| 2 | `FIELD verified` | hostname trùng thiết bị FIELD pcap đã có nhãn trong `device_labels_verified.csv` |
| 3 | `luật hostname` | luật `Code/pipeline/device_list.csv` (luật A của 05) cho đủ make/type/model; model mâu thuẫn make thì bỏ |
| 4 | `DHCP MSFT` | máy Windows không khớp luật nào → `WINDOWS_FALLBACK` (không đoán hãng) |
| — | `bỏ: <lý do>` | không hostname / thiếu thông tin → không vào train, không vào test |

OS lấy từ DHCP VCI (`MSFT` → Windows, `android-dhcp` → Android), Apple → iOS, điện thoại/tablet
hãng khác → Android; thiết bị FIELD cũ
dùng đúng bảng `FIELD_OS` của 05. `canonical_device` = hostname, nên các MAC ngẫu nhiên cùng
hostname được gom thành một thiết bị.

`device_labels_router.csv` được ghi lại mỗi lần chạy để xem nhãn đã dùng; dòng `verified=1` giữ nguyên.

In [ ]:
import ast
import types

sys.path.insert(0, str(ROOT / "Code" / "pipeline"))
try:
    import hostname_catalog
except ImportError:
    # hostname_data kéo sklearn vào chỉ để hostname_catalog lấy MODEL_ALIAS: nạp riêng hằng đó.
    _source = (ROOT / "Code" / "pipeline" / "hostname_data.py").read_text(encoding="utf-8")
    _node = next(n for n in ast.parse(_source).body if isinstance(n, ast.Assign)
                 and any(getattr(t, "id", "") == "MODEL_ALIAS" for t in n.targets))
    _stub = types.ModuleType("hostname_data")
    _stub.MODEL_ALIAS = ast.literal_eval(_node.value)
    sys.modules["hostname_data"] = _stub
    import hostname_catalog
from hostname_support import normalize_hostname

CATALOG_PATH = ROOT / "Code" / "pipeline" / "device_list.csv"
_catalog_rows, _ = hostname_catalog.load_catalog(CATALOG_PATH)
CATALOG_RULES = [(re.compile(pattern), head, label) for pattern, head, label in hostname_catalog.build_rules(
    {head: hostname_catalog.order_rows(_catalog_rows, head) for head in LABEL_COLS})]
CATALOG_MODEL_MAKE = defaultdict(set)
for _device in hostname_catalog.read_devices(CATALOG_PATH):
    if _device["make"] and _device[MODEL_HEAD]:
        CATALOG_MODEL_MAKE[_device[MODEL_HEAD]].add(_device["make"])


def notebook_literal(path, name):
    """Hằng literal trong một notebook khác (vd. FIELD_OS của 05), đọc bằng AST, không chạy notebook."""
    for cell in json.loads(path.read_text(encoding="utf-8"))["cells"]:
        source = "".join(cell.get("source", []))
        if cell["cell_type"] == "code" and f"{name} = " in source:
            for node in ast.parse(source).body:
                if isinstance(node, ast.Assign) and any(getattr(t, "id", "") == name for t in node.targets):
                    return ast.literal_eval(node.value)
    raise KeyError(f"không thấy {name} trong {path}")


FIELD_OS = notebook_literal(ROOT / "Code" / "pipeline" / "train_model" / "05_train.ipynb", "FIELD_OS")

verified_labels = pd.read_csv(LABELS_PATH, keep_default_na=False)
FIELD_NOTE = re.compile(r"field capture — (.+?) \(FIELD\)")
host_to_device = {}
for _, r in verified_labels.iterrows():
    m = FIELD_NOTE.search(str(r.get("review_note", "")))
    if m:
        host_to_device[m.group(1).strip().casefold()] = r["canonical_device"]
label_of = verified_labels.set_index("canonical_device")[LABEL_COLS]


def guess_os(vci, make="", type_=""):
    if vci.startswith("MSFT"):
        return "Windows"
    if vci.startswith("android-dhcp"):
        return "Android"
    if make == "Apple":
        return "iOS"
    return "Android" if type_ in ("Smartphone", "Tablet") else ""   # điện thoại không phải Apple


def catalog_label(hostname):
    name = normalize_hostname(hostname)
    hit = {}
    for pattern, head, label in CATALOG_RULES:
        if head not in hit and pattern.fullmatch(name):
            hit[head] = label
    return {head: label for head, label in hit.items() if label != UNKNOWN_LABEL}


def auto_label(hostname, vci):
    """(nhãn, nguồn) hoặc (None, lý do bỏ)."""
    if not hostname:
        return None, "bỏ: không có hostname"
    known = host_to_device.get(hostname.casefold())
    if known:
        labels = label_of.loc[known].to_dict()
        os_name = FIELD_OS.get(known) or guess_os(vci, labels["make"], labels["type"])
        if not os_name:
            return None, f"bỏ: không biết OS của {known}"
        return {"canonical_device": known, **labels, "os": os_name}, "FIELD verified"
    hit = catalog_label(hostname)
    if all(hit.get(head) for head in LABEL_COLS):
        makes = CATALOG_MODEL_MAKE.get(hit[MODEL_HEAD], {hit["make"]})
        if hit["make"] not in makes:
            return None, f"bỏ: luật hostname mâu thuẫn ({hit[MODEL_HEAD]} không thuộc {hit['make']})"
        os_name = guess_os(vci, hit["make"], hit["type"])
        if not os_name:
            return None, "bỏ: không đoán được OS"
        return {"canonical_device": hostname, **hit, "os": os_name}, "luật hostname"
    if WINDOWS_FALLBACK and vci.startswith("MSFT"):
        return {"canonical_device": hostname, **WINDOWS_FALLBACK, "os": "Windows"}, "DHCP MSFT"
    return None, "bỏ: luật hostname không đủ make/type/model"


def first_present(values):
    clean = [v for v in values if v and v != MISSING]
    return Counter(clean).most_common(1)[0][0] if clean else ""


LABEL_FIELDS = ["canonical_device", "make", "type", MODEL_HEAD, "os", "verified", "note"]
INFO_FIELDS = ["mac", "hostname", "dhcp_vci", "mdns_model", "mac_random", "mac_oui",
               "n_windows", "first_seen", "last_seen", "label_source"]

previous = pd.DataFrame(columns=["mac", *LABEL_FIELDS])
if LABELS_ROUTER.exists():
    previous = pd.read_csv(LABELS_ROUTER, dtype=str, keep_default_na=False)
manual = previous[previous["verified"].str.strip().str.casefold().isin(VERIFIED_VALUES)].set_index("mac")

host_vci = {h: first_present(g["dhcp_vci"]) for h, g in windows.groupby("hostname")}
info = []
for mac, g in windows.groupby("mac"):
    hostname = first_present(g["hostname"])
    vci = host_vci.get(hostname, "") or first_present(g["dhcp_vci"])
    row = {
        "mac": mac, "hostname": hostname, "dhcp_vci": vci,
        "mdns_model": first_present(g["serve_mdns_model"]),
        "mac_random": mac_is_random(mac), "mac_oui": mac[:8],
        "n_windows": len(g),
        "first_seen": g["start"].min().strftime("%Y-%m-%d %H:%M") if g["start"].notna().any() else "",
        "last_seen": g["end"].max().strftime("%Y-%m-%d %H:%M") if g["end"].notna().any() else "",
        **{col: "" for col in LABEL_FIELDS},
    }
    if mac in manual.index:
        row.update({col: str(manual.loc[mac, col]).strip() for col in LABEL_FIELDS if col in manual.columns})
        row["label_source"] = "nhãn tay"
    else:
        labels, source = auto_label(hostname, vci)
        row.update(labels or {})
        row["label_source"] = source
    info.append(row)
inventory = pd.DataFrame(info)[INFO_FIELDS + LABEL_FIELDS].astype(str)

gone = sorted(set(manual.index) - set(inventory["mac"]))
if gone:
    print(f"CẢNH BÁO: {len(gone)} MAC nhãn tay không còn observation: {gone}")

### 4b. Kiểm tra nhãn

- Nhãn **tay** sai (thiếu cột, OS lạ, `make=Unknown`, khác bản verified đã có) → dừng để bạn sửa.
- Nhãn **tự động** khác bản verified đã có hoặc thuộc `device_labels_excluded.csv` → bỏ MAC đó, ghi lý do.
- Cùng `canonical_device` mà các MAC mang nhãn khác nhau → dừng.

In [ ]:
is_manual = inventory["label_source"].eq("nhãn tay")
excluded_devices = set(pd.read_csv(EXCLUDED_PATH, keep_default_na=False)["canonical_device"])
need = ["canonical_device", *LABEL_COLS, "os"]
problems = []

for i, r in inventory[is_manual].iterrows():
    missing = [c for c in need if not r[c]]
    if missing:
        problems.append(f"{r.mac}: nhãn tay thiếu {missing}")
    if r.os and r.os not in OS_CLASSES:
        problems.append(f"{r.mac}: os={r.os!r} không thuộc {OS_CLASSES}")
    if r.make.casefold() == "unknown":
        problems.append(f"{r.mac}: make=Unknown bị cấm — chưa biết hãng thì bỏ verified")

for i, r in inventory[inventory["canonical_device"].ne("")].iterrows():
    reason = None
    if r.canonical_device in excluded_devices:
        reason = f"{r.canonical_device} nằm trong device_labels_excluded.csv"
    elif r.canonical_device in label_of.index:
        theirs = label_of.loc[r.canonical_device].to_dict()
        mine = {c: r[c] for c in LABEL_COLS}
        if mine != theirs:
            reason = f"nhãn {mine} khác bản verified {theirs}"
    if reason and is_manual[i]:
        problems.append(f"{r.mac}: {reason}")
    elif reason:
        inventory.loc[i, need] = ""
        inventory.loc[i, "label_source"] = f"bỏ: {reason}"

has_label = inventory["canonical_device"].ne("")
per_device = inventory[has_label].groupby("canonical_device")[[*LABEL_COLS, "os"]].nunique()
for device, row in per_device[per_device.gt(1).any(axis=1)].iterrows():
    problems.append(f"{device}: các MAC mang nhãn khác nhau ở {row[row.gt(1)].index.tolist()}")

inventory.sort_values(["label_source", "canonical_device", "mac"]).to_csv(
    LABELS_ROUTER, index=False, encoding="utf-8-sig")
if problems:
    print("\n".join(problems))
assert not problems, f"{len(problems)} lỗi nhãn tay — sửa {LABELS_ROUTER.name} rồi chạy lại"

labeled = inventory[has_label]
if labeled.empty:
    raise SystemExit("Không MAC nào gán được nhãn — xem cột label_source trong " + str(LABELS_ROUTER))
device_labels = labeled.groupby("canonical_device")[[*LABEL_COLS, "os", "label_source"]].first()
mac_device = labeled.set_index("mac")["canonical_device"]

print(f"{len(labeled)}/{len(inventory)} MAC có nhãn → {len(device_labels)} thiết bị; "
      f"bảng nhãn đã dùng: {rel(LABELS_ROUTER)}")
display(inventory.assign(nguồn=inventory["label_source"].str.replace(r"^bỏ:.*", "bỏ", regex=True))
        .groupby("nguồn").agg(MAC=("mac", "size"), thiết_bị=("canonical_device", lambda s: s[s.ne("")].nunique())))
display(device_labels)
dropped = inventory[~has_label]
print(f"\n{len(dropped)} MAC không có nhãn (không vào train/test):")
display(dropped[["mac", "hostname", "dhcp_vci", "n_windows", "label_source"]])

## 5. Gắn nhãn cửa sổ và làm sạch

In [ ]:
windows["canonical_device"] = windows["mac"].map(mac_device)
unlabeled = windows["canonical_device"].isna()
few_sources = windows["n_sources"] < MIN_SOURCES
print(f"không có nhãn: {int(unlabeled.sum())} cửa sổ / {windows.loc[unlabeled, 'mac'].nunique()} MAC")
print(f"< {MIN_SOURCES} nguồn (bỏ khỏi train): {int((~unlabeled & few_sources).sum())} cửa sổ có nhãn")

labeled_windows = windows[~unlabeled].copy()
for col in [*LABEL_COLS, "os"]:
    labeled_windows[col] = labeled_windows["canonical_device"].map(device_labels[col])

display(labeled_windows.groupby("canonical_device").agg(
    mac=("mac", "nunique"), cửa_sổ=("message_id", "size"),
    đủ_nguồn=("n_sources", lambda s: int((s >= MIN_SOURCES).sum())),
    type=("type", "first")).assign(nguồn_nhãn=lambda f: f.index.map(device_labels["label_source"]))
    .sort_values("cửa_sổ", ascending=False))

## 6. Chia train/test theo thiết bị

- Thiết bị **đã có trong FIELD pcap** (đã nằm trong train cũ) luôn ở `train`: giữ chúng ra
  làm test thì không còn là thiết bị chưa thấy.
- Thiết bị mới: chia theo `type`, mỗi `type` có ≥ 2 thiết bị mới thì `round(TEST_FRAC·n)`
  (tối thiểu 1) vào `test`. `type` chỉ có 1 thiết bị mới thì ở `train` (giữ ra thì lớp đó
  không có ai để học).
- Split ghi vào `device_split.csv` và **được giữ nguyên** ở các lần chạy sau; chỉ thiết bị
  mới được chia. Muốn chia lại từ đầu thì xoá file đó.

In [ ]:
raw_sessions = pd.read_parquet(RAW_SESSIONS_PATH)
pcap_field = set(raw_sessions.loc[raw_sessions["scenario"].eq(CAPTURE_SCENARIO)
                                  & ~raw_sessions["capture_file"].astype(str).str.startswith(ROUTER_PREFIX),
                                  "canonical_device"])

split = {}
if SPLIT_PATH.exists():
    frozen = pd.read_csv(SPLIT_PATH, keep_default_na=False)
    split = {r.canonical_device: (r.split, r.reason) for r in frozen.itertuples()}

rng = np.random.default_rng(SEED)
new_devices = [d for d in device_labels.index if d not in split]
for device in new_devices:
    if device in pcap_field:
        split[device] = ("train", "đã có trong FIELD pcap")
pending = [d for d in new_devices if d not in split]
for type_name, group in device_labels.loc[pending].groupby("type"):
    members = sorted(group.index)
    if len(members) < 2:
        split[members[0]] = ("train", f"type {type_name} chỉ có 1 thiết bị mới")
        continue
    n_test = max(1, int(round(TEST_FRAC * len(members))))
    test_members = set(rng.permutation(members)[:n_test])
    for d in members:
        split[d] = ("test", "held-out thiết bị") if d in test_members else ("train", "chia ngẫu nhiên")

forced = [d for d, (s, _) in split.items() if s == "test" and d in pcap_field]
assert not forced, f"thiết bị FIELD pcap không được nằm ở test: {forced}"
pd.DataFrame([{"canonical_device": d, "split": s, "reason": r} for d, (s, r) in sorted(split.items())]
             ).to_csv(SPLIT_PATH, index=False, encoding="utf-8-sig")

labeled_windows["split"] = labeled_windows["canonical_device"].map(lambda d: split[d][0])
assert labeled_windows.groupby("canonical_device")["split"].nunique().eq(1).all()


def cap_windows(g, limit):
    """Giữ `limit` cửa sổ trải đều theo thời gian, không lấy dồn một ngày."""
    g = g.sort_values("start")
    if len(g) <= limit:
        return g
    return g.iloc[np.unique(np.linspace(0, len(g) - 1, limit).round().astype(int))]


train_pool = labeled_windows[labeled_windows["split"].eq("train") & (labeled_windows["n_sources"] >= MIN_SOURCES)]
train_windows = pd.concat([cap_windows(g, MAX_WINDOWS_PER_DEVICE)
                           for _, g in train_pool.groupby("canonical_device")] or [train_pool])
test_windows = labeled_windows[labeled_windows["split"].eq("test")]

summary = pd.DataFrame({
    "thiết bị": labeled_windows.groupby("split")["canonical_device"].nunique(),
    "cửa sổ có nhãn": labeled_windows.groupby("split").size(),
})
summary["cửa sổ dùng"] = pd.Series({"train": len(train_windows), "test": len(test_windows)})
display(summary)
print("type × split (thiết bị):")
display(pd.crosstab(device_labels["type"], device_labels.index.map(lambda d: split[d][0])))

## 7. Bộ test giữ riêng

- `test_observations.jsonl`: observation gốc của thiết bị test (mọi cửa sổ, kể cả 1 nguồn —
  đo đúng cách runtime gặp chúng). Chạy được thẳng bằng `Code/test_release_observations.py --data`.
- `test_windows.parquet`: đúng 12 cột input (từ adapter chạy thật) + nhãn.
- `test_labels.csv`: MAC → nhãn, để chấm kết quả.

In [ ]:
test_ids = set(test_windows["message_id"])
(OBS_DIR / "test_observations.jsonl").write_text(
    "".join(json.dumps(o, ensure_ascii=False) + "\n" for o in observations if o["message_id"] in test_ids),
    encoding="utf-8")

test_frame = test_windows[["message_id", "mac", "canonical_device", "n_sources", *LABEL_COLS, "os"]].copy()
for col in serve.COLUMNS:
    test_frame[col] = test_windows[col if col in SHARED else f"serve_{col}"]
test_frame.to_parquet(OBS_DIR / "test_windows.parquet", index=False)
(test_windows.groupby("mac")[["canonical_device", *LABEL_COLS, "os"]].first()
 .to_csv(OBS_DIR / "test_labels.csv", encoding="utf-8-sig"))

labeled_windows.drop(columns=["start", "end"]).assign(
    used_in_train=labeled_windows["message_id"].isin(set(train_windows["message_id"]))
).to_parquet(OBS_DIR / "windows_labeled.parquet", index=False)
print(f"test: {len(test_frame)} cửa sổ / {test_frame['canonical_device'].nunique()} thiết bị → {rel(OBS_DIR)}")

### 7b. (Tuỳ chọn) Đo một model release trên bộ test

Đặt `EVAL_MODEL` ở mục 1. Báo cáo mức cửa sổ (coverage, đúng trên phần đã trả lời) và mức
thiết bị (bỏ phiếu đa số trên các cửa sổ đã trả lời, cần ≥ 3 phiếu như `DeviceTracker`).

Đo hai lần: đủ 12 cột, và **bỏ hostname** (`dhcp_hostname = <missing>`). Nhãn tự động phần lớn
suy từ hostname, mà model cũng đọc hostname, nên lần đủ cột bị "tự chấm điểm" ở các thiết bị đó;
lần bỏ hostname mới cho thấy model nhận ra được bao nhiêu từ DHCP/TLS/DNS/mDNS.
Output release gộp `Smartphone`/`Tablet` thành `Smartphone/Tablet`, nên nhãn thật được gộp
tương ứng trước khi so.

In [ ]:
if EVAL_MODEL is not None and len(test_frame):
    import onnxruntime as ort
    from pipeline.hostname_support import run_onnx

    session = ort.InferenceSession(str(EVAL_MODEL))
    alias = {"Smartphone": "Smartphone/Tablet", "Tablet": "Smartphone/Tablet"}
    inputs = test_frame[list(serve.COLUMNS)]
    variants = {"đủ 12 cột": inputs, "bỏ hostname": inputs.assign(dhcp_hostname=MISSING)}
    report, preds = [], {}
    for variant, frame in variants.items():
        matrix = np.asarray(frame.to_numpy(), dtype=object)
        pred = preds[variant] = run_onnx(session, ["output"], {"input": matrix})[0].astype(str)
        for i, head in enumerate(LABEL_COLS):
            truth = test_frame[head].replace(alias) if head == "type" else test_frame[head]
            p = pd.Series(pred[:, i], index=test_frame.index)
            answered = p.ne(UNKNOWN_LABEL)
            votes = pd.DataFrame({"device": test_frame["canonical_device"], "pred": p,
                                  "truth": truth})[answered]
            dev = votes.groupby("device").agg(n=("pred", "size"),
                                              pred=("pred", lambda s: s.mode().iat[0]),
                                              truth=("truth", "first"))
            dev = dev[dev["n"] >= 3]
            report.append({
                "input": variant, "head": head,
                "coverage cửa sổ": round(answered.mean(), 3),
                "đúng / đã trả lời": round((p[answered] == truth[answered]).mean(), 3) if answered.any() else np.nan,
                "thiết bị đủ 3 phiếu": f"{len(dev)}/{test_frame['canonical_device'].nunique()}",
                "đúng mức thiết bị": round((dev["pred"] == dev["truth"]).mean(), 3) if len(dev) else np.nan,
            })
    display(pd.DataFrame(report).set_index(["input", "head"]))
    pred = preds["đủ 12 cột"]
    wrong = test_frame.assign(**{f"pred_{h}": pred[:, i] for i, h in enumerate(LABEL_COLS)})
    display(wrong.groupby("canonical_device")[[*LABEL_COLS, *[f"pred_{h}" for h in LABEL_COLS]]]
            .agg(lambda s: s.mode().iat[0]))
else:
    print("EVAL_MODEL = None — bỏ qua đo model")

## 8. Ghi vào dataset train (`APPLY = True`)

Backup `sessions.parquet`, `sessions_verified.parquet`, `device_labels_verified.csv` vào
`Data/router_obs/backup/<thời điểm>/`, rồi:

1. `sessions.parquet`: xoá mọi dòng `router/` cũ, thêm session train mới (cùng cột, cùng
   kiểm tra lệch cột như 04).
2. `device_labels_verified.csv`: thêm thiết bị router mới; bỏ thiết bị router không còn session.
3. `build_verified_dataset()` dựng lại `sessions_verified.parquet`.
4. `dhcp_features_router.csv` (hostname DHCP của cửa sổ train, chỉ thiết bị có nhãn từ
   `TRUSTED_SOURCES`) và `device_os_router.csv` (nhãn OS) cho 05.

In [ ]:
ROUTER_NOTE = "router MQTT"

new_sessions = pd.DataFrame([{
    "session_id": f"{r.canonical_device}_router_{r.message_id}",
    "canonical_device": r.canonical_device,
    "scenario": CAPTURE_SCENARIO,
    "capture_file": ROUTER_PREFIX + r.message_id,
    "date": r.start.strftime("%Y-%m-%d") if pd.notna(r.start) else "",
    "mac": r.mac,
    **{c: getattr(r, c) for c in features},
    "n_sources": r.n_sources,
    "make": r.make, "type": r.type, "family": getattr(r, MODEL_HEAD),   # cột thô vẫn tên "family"
} for r in train_windows.itertuples()]).set_index("session_id").sort_index()

missing_cols = set(raw_sessions.columns) - set(new_sessions.columns)
extra_cols = set(new_sessions.columns) - set(raw_sessions.columns)
assert not missing_cols and not extra_cols, (
    f"Lệch cột so với {RAW_SESSIONS_PATH.name}: thiếu {missing_cols}, thừa {extra_cols}")

train_devices = set(new_sessions["canonical_device"])
hosts = train_windows.groupby("canonical_device")["hostname"].agg(
    lambda s: ", ".join(sorted({h for h in s if h})) or "không hostname")
labels_df = pd.read_csv(LABELS_PATH, keep_default_na=False)
is_router_row = labels_df["review_note"].astype(str).str.startswith(ROUTER_NOTE)
labels_df = labels_df[~(is_router_row & ~labels_df["canonical_device"].isin(train_devices))]
new_label_rows = pd.DataFrame([{
    "canonical_device": d, **device_labels.loc[d, LABEL_COLS].to_dict(),
    "review_note": f"{ROUTER_NOTE} — {hosts[d]} — {device_labels.loc[d, 'label_source']} (FIELD)",
} for d in sorted(train_devices - set(labels_df["canonical_device"]))])

train_ids = set(train_windows["message_id"])
dhcp_rows = [{
    "mac": obs["mac"], "device": mac_device[obs["mac"]],
    "capture_file": ROUTER_PREFIX + obs["message_id"], "proto": "dhcp",
    "msg_type": d.get("message_name"), "hostname": d.get("hostname") or "",
    "vendor_class_id": d.get("vendor_class_id") or "",
    "param_req_list": ",".join(str(v) for v in d["parameter_request_list"]),
    "ts": d.get("timestamp"),
} for obs in observations if obs["message_id"] in train_ids
  and device_labels.loc[mac_device[obs["mac"]], "label_source"] in TRUSTED_SOURCES
  for d in client_dhcp(obs) if (d.get("hostname") or "").strip()]

print(f"session router train: {len(new_sessions)} / {len(train_devices)} thiết bị; "
      f"thiết bị mới vào taxonomy: {len(new_label_rows)}; dòng hostname DHCP: {len(dhcp_rows)}")
display(new_label_rows)

if not APPLY:
    print("\nAPPLY = False — chưa ghi gì vào dataset train. Xem lại các bảng trên, rồi đặt APPLY = True.")
else:
    backup = OBS_DIR / "backup" / STAMP
    backup.mkdir(parents=True, exist_ok=True)
    for path in (RAW_SESSIONS_PATH, VERIFIED_OUT, LABELS_PATH):
        if path.exists():
            shutil.copy2(path, backup / path.name)
    print("backup:", rel(backup))

    kept = raw_sessions[~raw_sessions["capture_file"].astype(str).str.startswith(ROUTER_PREFIX)]
    combined = pd.concat([kept, new_sessions[raw_sessions.columns]])
    assert not combined.index.duplicated().any(), "session_id trùng sau khi gộp"
    combined.sort_index().to_parquet(RAW_SESSIONS_PATH)
    print(f"đã ghi {RAW_SESSIONS_PATH.name}: {len(combined)} session "
          f"(bỏ {len(raw_sessions) - len(kept)} dòng router cũ, thêm {len(new_sessions)})")

    pd.concat([labels_df, new_label_rows], ignore_index=True).to_csv(LABELS_PATH, index=False)
    result = build_verified_dataset(source=RAW_SESSIONS_PATH, labels_path=LABELS_PATH,
                                    excluded_path=EXCLUDED_PATH, output=VERIFIED_OUT)

    pd.DataFrame(dhcp_rows).to_csv(ROUTER_DHCP_CSV, index=False)
    device_labels.loc[sorted(train_devices), ["os"]].to_csv(ROUTER_OS_CSV)
    print(f"đã ghi {ROUTER_DHCP_CSV.name}, {ROUTER_OS_CSV.name}")
    display(result[result["capture_file"].astype(str).str.startswith(ROUTER_PREFIX)]
            .groupby("canonical_device").size().rename("session router"))

## Xong

Bước tiếp:

1. `train_model/05_train.ipynb` — thiết bị router là lớp thật (scenario FIELD); 05 đọc
   `device_os_router.csv` cho head OS và `dhcp_features_router.csv` cho hostname expert.
2. Đo bản mới trên **bộ test giữ riêng**: đặt `EVAL_MODEL` ở mục 1 và chạy lại 7b, hoặc
   `python Code/test_release_observations.py --data Data/router_obs/test_observations.jsonl`.
   So với release đang chạy trên cùng bộ test — đây mới là phép đo thiết bị chưa thấy.
3. Đạt thì `09 → 12 → 13`.